# KubeCon India 2026 — Platform + App Setup (AWS / Crossplane track)

The notebook form of [`setup-with-xp.sh`](setup-with-xp.sh) — run it **after**
[`00_init-with-xp.ipynb`](00_init-with-xp.ipynb) has bootstrapped the cluster
(k3d + Crossplane + KubeVela).

It applies the platform-team building blocks (the **How**) and deploys the developer
Application (the **What**) to prove the one-interface model end to end — and it
reuses the repo's reusable helpers in [`../../../scripts/`](../../../scripts/) and the
per-app `build-image.sh` scripts, so the notebook and `setup-with-xp.sh` never drift.

## Steps
1. Apply the platform building blocks — S3 XRD + Composition, then the `bucket` backing (`bucket-xp.cue`)
2. Build + push the app images (product-catalog **and** bucket-browser)
3. Create AWS credentials in the app namespaces (dev / staging / prod)
4. Deploy the Application (`product-catalog.yaml`) — multi-env workflow
5. Verify

**Run the cells in order.** Each `%%bash` cell is self-contained.


## Step 1: Apply platform building blocks (the How)

Apply the Crossplane S3 **XRD** then its **Composition** (order matters — the
composition references the composite type), then `vela def apply` the **`bucket-xp`**
ComponentDefinition the developer Application claims. (The function the composition
pipeline uses was installed by the init notebook.)

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"   # notebook runs from aws-setup/
source "$REPO_ROOT/scripts/common.sh"

print_step "Applying Crossplane S3 definition + composition"
kubectl apply -f "$REPO_ROOT/platform/crossplane/s3/definition.yaml"
kubectl apply -f "$REPO_ROOT/platform/crossplane/s3/composition.yaml"
print_success "S3 definition + composition applied"

print_step "Applying the 'bucket' ComponentDefinition (Crossplane backing)"
vela def apply "$REPO_ROOT/platform/kubevela/components/bucket-xp.cue"
print_success "'bucket' ComponentDefinition installed"


## Step 2: Build + push the app images

Build both app images and push them to the local k3d registry: the
**product-catalog** API and the **bucket-browser** web UI (the two `webservice`
components in the Application). The apps are track-agnostic — the same images run on
AWS or GCP; only `STORAGE_PROVIDER` + the mounted secret differ.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"

bash "$REPO_ROOT/apps/product-catalog/build-image.sh"
bash "$REPO_ROOT/apps/bucket-browser/build-image.sh"


## Step 3: AWS credentials in the app namespaces

App pods mount an `aws-credentials` secret to reach S3 (this is the *app's* runtime
credential, distinct from Crossplane's). Load `.env.aws`, then create the secret in
each environment namespace (creating the namespace if absent).

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source "$REPO_ROOT/scripts/load-aws-env.sh"
source "$REPO_ROOT/scripts/create-aws-secret.sh"

load_aws_env "$PWD/.env.aws"
for ns in dev staging prod; do
    create_aws_secret --create-namespace "$ns" aws-credentials
done


## Step 4: Deploy the Application

Submit the developer Application. Its multi-env workflow auto-deploys to **dev** (and
runs the functional API tests), then **suspends** for manual approval before staging
and prod — so this returns once dev is rolling. Resume later with:
`vela workflow resume product-catalog`.

The Application has three components: the `product-catalog-api`, the read-only
`bucket-browser` UI, and the `bucket` claim (resolved by the `bucket-xp` backing → an
S3 bucket per environment).

In [ ]:
%%bash
set -euo pipefail
DEMO_DIR="$(cd .. && pwd)"          # the shared Application lives in the parent demo dir
source "$(cd ../../.. && pwd)/scripts/common.sh"

vela up -f "$DEMO_DIR/kubevela/product-catalog.yaml"
print_success "Application submitted (workflow deploys dev, then suspends for approval)"


## Step 5: Verify

Check the Application status and the running workloads. The functional `request` steps
in the workflow already exercised the API against its S3 bucket; the
`bucket-browser` UI lets you *see* the objects those test products created.

In [ ]:
%%bash
set -euo pipefail
vela status product-catalog || true
echo
kubectl get pods,hpa -n dev
echo
echo "The dev S3 bucket the claim provisioned:"
kubectl get bucket.s3.aws.upbound.io -A || true
echo
echo "Open the bucket-browser UI to see the objects (run in a separate terminal):"
echo "  kubectl -n dev port-forward svc/bucket-browser 8080:8080   # http://localhost:8080"


## Setup complete

The Crossplane track is fully deployed: the `bucket-xp` backing is installed, both app
images are built, and the `product-catalog` Application is rolling out (API +
bucket-browser + an S3 bucket per environment).

### Next steps
- Resume the multi-env workflow: `vela workflow resume product-catalog`.
- Browse the objects: `kubectl -n dev port-forward svc/bucket-browser 8080:8080` → http://localhost:8080.
- Swap the backing (same claim): `vela def apply ../../../platform/kubevela/components/bucket-ack.cue` (ACK),
  or run the GCP track in [`../gcp-setup/`](../gcp-setup/) (KCC).

### Teardown
- ACK track only: `./teardown-with-ack.sh` (empties the S3 buckets first).
- Then delete the cluster: `../cleanup.sh`.
